# Find Variable Sites from RSV F Protein Alignment

This notebook identifies mutations in RSV F protein sequences by comparing them to the DMS strain RSV Long sequence. The alignment of RSV F protein sequences for RSV-A and RSV-B was generated on October 27, 2025 from the RSV Nextstrain workflow: https://github.com/nextstrain/rsv.

In [ ]:
# Parameters - will be overridden by papermill
strain = "RSV-A"
alignment_file = "data/RSV_F_seqs/RSV-A_with_ref.fasta"
output_file = "results/sequence_variation/RSV-A_sequence_variations_with_effects.csv"
cell_entry_file = "results/summaries/cell_entry.csv"
min_mutation_count = 5
reference_name = "RSV_Long_F"  # Name of reference sequence in alignment
analysis_description = "Differences from Strain Used in Deep Mutational Scanning"
mutation_identified_relative_to = "RSV_Long_F"
effects_calculated_relative_to = "RSV_Long_F"

In [ ]:
import pandas as pd
import numpy as np
from Bio import SeqIO
from collections import Counter
import os

## Configuration

In [ ]:
print(f"Analyzing strain: {strain}")
print(f"Analysis type: {analysis_description}")
print(f"Input alignment: {alignment_file}")
print(f"Reference sequence: {reference_name}")
print(f"Differences identified relative to: {mutation_identified_relative_to}")
print(f"Effects calculated relative to: {effects_calculated_relative_to}")
print(f"Cell entry data: {cell_entry_file}")
print(f"Output file: {output_file}")
print(f"Minimum mutation count threshold: {min_mutation_count}")

## Define function to analyze variable sites

In [ ]:
def analyze_variable_sites_detailed(fasta_file, reference_name, min_mutation_count=5):
    """
    Analyze variable sites in a multiple sequence alignment.
    
    Parameters:
    -----------
    fasta_file : str
        Path to FASTA alignment file
    reference_name : str
        Name/ID of the reference sequence to use for comparison
    min_mutation_count : int
        Minimum number of sequences with a mutation for it to be reported
        
    Returns:
    --------
    pd.DataFrame
        DataFrame with columns: site, wildtype, mutant, mutation_count, mutation_type
    """
    # Parse all sequences
    sequences = list(SeqIO.parse(fasta_file, 'fasta'))
    
    # Find the reference sequence by name
    reference = None
    for seq in sequences:
        if seq.id == reference_name:
            reference = seq
            break
    
    if reference is None:
        raise ValueError(f"Reference sequence '{reference_name}' not found in alignment")
    
    ref_name = reference.id
    ref_seq = str(reference.seq)
    
    print(f"Reference sequence: {ref_name}")
    print(f"Reference length: {len(ref_seq)} amino acids")
    print(f"Total sequences in alignment: {len(sequences)}")
    
    # Track mutations at each position
    variable_sites = []
    
    # Iterate through each position in the reference
    for pos in range(len(ref_seq)):
        site = pos + 1  # 1-based numbering
        ref_aa = ref_seq[pos]
        
        # Skip stop codons and gaps in reference
        if ref_aa == '*' or ref_aa == '-':
            continue
        
        # Collect all amino acids at this position across sequences
        mutations = []
        for seq in sequences:
            # Skip the reference itself
            if seq.id == reference_name:
                continue
                
            if pos < len(seq.seq):
                seq_aa = str(seq.seq[pos])
                # Only count if different from reference and not a stop or gap
                if seq_aa != ref_aa and seq_aa != '*' and seq_aa != '-':
                    mutations.append(seq_aa)
        
        # Count mutation frequencies
        if mutations:
            mutation_counts = Counter(mutations)
            
            # Report mutations meeting the threshold
            for mutant_aa, count in mutation_counts.items():
                if count >= min_mutation_count:
                    variable_sites.append({
                        'site': site,
                        'wildtype': ref_aa,
                        'mutant': mutant_aa,
                        'mutation_count': count,
                        'mutation_type': f"{ref_aa}→{mutant_aa}"
                    })
    
    # Create DataFrame and sort
    df = pd.DataFrame(variable_sites)
    if len(df) > 0:
        df = df.sort_values(['site', 'mutation_count'], ascending=[True, False])
    
    print(f"\nFound {len(df)} sequence variations at {df['site'].nunique()} sites")
    print(f"(appearing in at least {min_mutation_count} sequences)")
    
    return df

## Analyze alignment

In [ ]:
# Run analysis
variable_sites = analyze_variable_sites_detailed(
    alignment_file,
    reference_name,
    min_mutation_count=min_mutation_count
)

# Display first rows
print(f"\nFirst 20 sequence variations for {strain}:")
display(variable_sites.head(20))

## Load cell entry data and calculate effects

In [ ]:
# Load cell entry effects
cell_entry = pd.read_csv(cell_entry_file)
print(f"Loaded cell entry data: {len(cell_entry)} mutations")
print(f"Sites range: {cell_entry['site'].min()} - {cell_entry['site'].max()}")

# Build lookup dictionary: effects[site][amino_acid] = cell_entry_effect
effects = {}
site_info = {}  # Store sequential_site and region for each site

for _, row in cell_entry.iterrows():
    site = row['site']
    if site not in effects:
        effects[site] = {}
        site_info[site] = {
            'sequential_site': row['sequential_site'],
            'region': row['region']
        }
    effects[site][row['mutant']] = row['cell entry']

# Calculate effect for each sequence variation
variation_effects = []
variations_with_dms_data = 0

for _, var in variable_sites.iterrows():
    site = var['site']
    wt = var['wildtype']  # Wildtype from DMS strain
    mut = var['mutant']   # Amino acid in natural sequence
    
    # Check if we have DMS measurement for this mutant at this site
    if site in effects and mut in effects[site]:
        # Get the DMS cell entry effect for this mutation
        # (DMS effects are already normalized relative to wildtype)
        cell_entry_effect = effects[site][mut]
        variations_with_dms_data += 1
        
        variation_effects.append({
            'site': site,
            'wildtype': wt,
            'mutant': mut,
            'mutation_type': var['mutation_type'],
            'mutation_count': var['mutation_count'],
            'cell entry': cell_entry_effect,
            'sequential_site': site_info[site]['sequential_site'],
            'region': site_info[site]['region']
        })
    else:
        # Include variation even without DMS data, with NaN for effect columns
        variation_effects.append({
            'site': site,
            'wildtype': wt,
            'mutant': mut,
            'mutation_type': var['mutation_type'],
            'mutation_count': var['mutation_count'],
            'cell entry': np.nan,
            'sequential_site': site_info.get(site, {}).get('sequential_site', np.nan),
            'region': site_info.get(site, {}).get('region', pd.NA)
        })

with_effects = pd.DataFrame(variation_effects)

print(f"\nTotal {strain} sequence variations: {len(with_effects)}")
print(f"Variations with DMS data: {variations_with_dms_data}")
print(f"Variations without DMS data: {len(with_effects) - variations_with_dms_data}")
print(f"At {with_effects['site'].nunique()} sites")
print(f"\nCoverage: {variations_with_dms_data / len(with_effects) * 100:.1f}% of sequence variations have DMS data")

## Summary statistics

Display summary of variations with effects

In [ ]:
# Display first rows
print(f"\nFirst 20 sequence variations for {strain}:")
display(with_effects.head(20))

# Filter to variations with DMS data for additional statistics
with_dms = with_effects.dropna(subset=['cell entry'])

if len(with_dms) > 0:
    print(f"\n\nCell Entry Effect Statistics (based on {len(with_dms)} variations with DMS data):")
    print("=" * 60)
    print(f"Mean effect: {with_dms['cell entry'].mean():.3f}")
    print(f"Median effect: {with_dms['cell entry'].median():.3f}")
    print(f"Std deviation: {with_dms['cell entry'].std():.3f}")
    print(f"Effect range: {with_dms['cell entry'].min():.3f} to {with_dms['cell entry'].max():.3f}")
    
    print(f"\nMost variable sites:")
    top_sites = with_effects.groupby('site').size().sort_values(ascending=False).head(10)
    for site, count in top_sites.items():
        wt = with_effects[with_effects['site'] == site]['wildtype'].iloc[0]
        print(f"  Site {site} ({wt}): {count} different mutations observed")
    
    print(f"\nMost common sequence variations with highest effects:")
    top_with_effects = with_dms.nlargest(10, 'cell entry')[['site', 'mutation_type', 'mutation_count', 'cell entry', 'region']]
    display(top_with_effects)
else:
    print(f"\nNo DMS data available for these sequence variations.")

## Save results

In [ ]:
# Create output directory if needed
os.makedirs(os.path.dirname(output_file), exist_ok=True)

# Save to CSV
with_effects.to_csv(output_file, index=False)
print(f"\nSaved results to: {output_file}")
print(f"Total variations saved: {len(with_effects)}")
print(f"  - With DMS data: {len(with_effects.dropna(subset=['cell entry']))}")
print(f"  - Without DMS data: {len(with_effects[with_effects['cell entry'].isna()])}")